In [69]:
import torch

print("cuda available" if torch.cuda.is_available() else "cuda unavailable")
print("gpu ready" if torch.cuda.device_count() else "only cpu")

cuda available
gpu ready


In [70]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from autoencoder.dataset import TrackDataset, build_label_vocab
from autoencoder.model import TrackAutoencoder, reconstruction_loss

In [71]:
data_path = Path("..") / "data/engineered/training_pop30_genres.parquet"
df = pd.read_parquet(data_path)
print(f"Loaded {len(df):,} tracks from {data_path}")

Loaded 1,862,314 tracks from ../data/engineered/training_pop30_genres.parquet


In [72]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1862314 entries, 0 to 1862313
Data columns (total 24 columns):
 #   Column            Dtype   
---  ------            -----   
 0   track_rowid       int64   
 1   track_name        string  
 2   artist_name       string  
 3   artist_rowid      int64   
 4   album_rowid       int64   
 5   album_name        string  
 6   album_type        category
 7   _label            category
 8   release_season    category
 9   release_year      int32   
 10  time_signature    uint8   
 11  tempo             float32 
 12  key               uint8   
 13  mode              uint8   
 14  danceability      float32 
 15  energy            float32 
 16  loudness          float32 
 17  speechiness       float32 
 18  acousticness      float32 
 19  instrumentalness  float32 
 20  liveness          float32 
 21  valence           float32 
 22  explicit          bool    
 23  duration_ms       float64 
dtypes: bool(1), category(3), float32(9), float64(1), i

## Test-train split

In [73]:
ntrain = int(0.8 * len(df)); nvalid = len(df) - ntrain
_shuffled = df.sample(frac=1, replace=False, ignore_index=True, random_state=0)
dft = _shuffled.loc[:ntrain].reset_index()
dfv = _shuffled.loc[ntrain + 1: ].reset_index()

## Indexing labels

In [6]:
# Build label vocabulary from training data
label_to_idx = build_label_vocab(dft)
print(f"Number of unique labels: {len(label_to_idx)}")

Number of unique labels: 8363


## Feature standardization

In [74]:
def minmax(s: pd.Series, r: pd.Series | None=None, vrange:tuple[int, int]=(0, 1)) -> pd.Series:
    if r is None:
        r = s
    a, b = vrange
    return a + (s - r.min()) * (b - a) / (r.max() - r.min())

def zscore(s: pd.Series, r: pd.Series | None=None, mode: str = "mean"):
    if r is None:
        r = s
    if mode == "mean":
        m = r.mean()
    elif mode == "median":
        m = r.median()
    else:
        raise ValueError()
    return (s - m) / r.std()


ZSCORE_COLS = ["tempo", "duration_ms", "danceability", "energy", "loudness", "liveness", "valence"]
MINMAX_COLS = ["release_year", "speechiness", "acousticness", "instrumentalness"]

for col in ZSCORE_COLS:
  dfv[col] = zscore(dfv[col], dft[col], mode="median")
  dft[col] = zscore(dft[col], mode="median")

for col in MINMAX_COLS:
  dfv[col] = minmax(dfv[col], dft[col], vrange=(-1., 1.))
  dft[col] = minmax(dft[col], vrange=(-1., 1.))

In [76]:
if (_plot_train_correlations := False):
    features = [
        "tempo", "duration_ms", "release_year", "danceability", "energy",
        "loudness", "speechiness", "acousticness", "instrumentalness",
        "liveness", "valence",
    ]
    n_feat = len(features)
    fig, axes = plt.subplots(n_feat, n_feat, figsize=(12,12))
    
    for i in range(n_feat):
        for j in range(n_feat):
            ax = axes[i, j]
            if i == j:
                ax.hist(_df[features[i]], bins=30, color='steelblue', edgecolor='white')
            else:
                ax.hist2d(_df[features[j]], _df[features[i]], bins=30)
            
            if i == n_feat - 1:
                ax.set_xlabel(features[j], rotation=45, ha='right')
                ax.tick_params(axis='x', labelsize=7, rotation=45)
            else:
                ax.set_xticks([])
                
            if j == 0:
                ax.set_ylabel(features[i])
                ax.tick_params(axis='y', labelsize=7)
            else:
                ax.set_yticks([])
    
    plt.tight_layout()
    plt.show()